In [ ]:
dir.create("../../results/diablo/figures/", showWarnings = FALSE, recursive = TRUE)
options(repr.plot.width = 9, repr.plot.height = 6, repr.plot.res = 150)
knitr::opts_chunk$set(warning = FALSE, message = FALSE)

## Biological Background

The **PAM50 intrinsic subtypes** are the clinical standard for molecular classification
of breast cancer. The full taxonomy is broader, but the `breast.TCGA` demo subset used
here contains three labels only: Basal, Her2, and LumA. `LumA` here is the actual
Luminal A class, not a merged Luminal A/B group:

| Subtype | ER | HER2 | Key biology | Standard treatment |
|---------|----|------|------------|-------------------|
| Basal-like | − | − | TP53 mutations, BRCA1-like, high proliferation | Chemotherapy |
| HER2-enriched | − | + | ERBB2 amplification (17q12) | Trastuzumab + chemo |
| Luminal A | + | − | ESR1/FOXA1-driven, low proliferation | Endocrine therapy only |

The `breast.TCGA` dataset from the **mixOmics** package provides three omics layers
for 150 training samples across three subtypes (Basal, Her2, LumA):

- **mRNA**: 200 most discriminant transcripts (RNA-seq, log2 CPM)
- **miRNA**: 184 mature microRNAs (regulates mRNA post-transcriptionally)
- **proteomics**: 142 RPPA (Reverse Phase Protein Array) measurements

**Biological question**: What minimal multi-omics biomarker panel — spanning mRNA,
miRNA, and proteomics — best discriminates PAM50 breast cancer subtypes?

---

## Setup

In [ ]:
library(mixOmics)
library(ggplot2)
library(dplyr)
library(tidyr)
library(purrr)
library(pheatmap)
library(corrplot)
library(ggrepel)
library(RColorBrewer)

theme_set(theme_bw(base_size = 12))

# Consistent PAM50 colour palette — use across all DIABLO notebooks
subtype_colors <- c(
  Basal = "#E41A1C",   # red
  Her2  = "#FF7F00",   # orange
  LumA  = "#4DAF4A"    # green
)

set.seed(2024)

---

## 1. Load breast.TCGA Dataset

In [ ]:
# breast.TCGA is bundled in the mixOmics package.
# data.train / data.test split is pre-defined to prevent data leakage
# during model tuning in Notebook 3.
#
# Field names in this mixOmics version: mirna, mrna, protein, subtype
# NOTE: test set does not include a proteomics block.
#
# IMPORTANT: Unlike MOFA2 (features × samples), mixOmics uses
# SAMPLES × FEATURES orientation.

data(breast.TCGA, package = "mixOmics")

# Training data — all three omics blocks available
X_train <- list(
  mRNA       = breast.TCGA$data.train$mrna,
  miRNA      = breast.TCGA$data.train$mirna,
  proteomics = breast.TCGA$data.train$protein
)
Y_train <- breast.TCGA$data.train$subtype

# Test data — proteomics not included in this dataset split; mRNA + miRNA only
X_test <- list(
  mRNA  = breast.TCGA$data.test$mrna,
  miRNA = breast.TCGA$data.test$mirna
)
Y_test <- breast.TCGA$data.test$subtype

cat("Training dimensions:\n")
map(X_train, dim) |> bind_rows(.id = "block") |>
  setNames(c("block", "n_samples", "n_features")) |> print()

cat("\nTest dimensions (proteomics not available in test split):\n")
map(X_test, dim) |> bind_rows(.id = "block") |>
  setNames(c("block", "n_samples", "n_features")) |> print()

---

## 2. Missingness Audit: NA Values vs Missing Views

In [ ]:
# Missingness audit: distinguish NA values from missing whole views.
# NA counts can be zero inside available matrices while a full block is absent.

na_train <- map_dfr(names(X_train), function(block) {
  m <- X_train[[block]]
  data.frame(
    block = block,
    set = "Train",
    samples = nrow(m),
    features = ncol(m),
    na_count = sum(is.na(m)),
    na_pct = 100 * mean(is.na(m))
  )
})

na_test <- map_dfr(names(X_test), function(block) {
  m <- X_test[[block]]
  data.frame(
    block = block,
    set = "Test",
    samples = nrow(m),
    features = ncol(m),
    na_count = sum(is.na(m)),
    na_pct = 100 * mean(is.na(m))
  )
})

na_summary <- bind_rows(na_train, na_test)
print(na_summary)

all_blocks <- names(X_train)
block_availability <- data.frame(
  block = all_blocks,
  train_available = all_blocks %in% names(X_train),
  test_available = all_blocks %in% names(X_test)
) |>
  mutate(
    missing_views_in_test = !test_available,
    missing_pct_in_test = ifelse(missing_views_in_test, 100, 0)
  )

cat("\nBlock-level availability:\n")
print(block_availability)

---

## 3. Class Balance

In [ ]:
# Class imbalance causes biased classifiers. Check the distribution
# of PAM50 subtypes in both train and test sets.
# LumA is the most common subtype in TCGA (~40-50% of samples).

bind_rows(
  data.frame(set = "Train", subtype = Y_train),
  data.frame(set = "Test",  subtype = Y_test)
) |>
  count(set, subtype) |>
  group_by(set) |>
  mutate(pct = round(100 * n / sum(n), 1)) |>
  ggplot(aes(x = subtype, y = n, fill = subtype)) +
  geom_col(show.legend = FALSE) +
  geom_text(aes(label = paste0(n, "\n(", pct, "%)")), vjust = -0.3, size = 3.5) +
  facet_wrap(~ set) +
  scale_fill_manual(values = subtype_colors) +
  ylim(0, max(table(Y_train)) * 1.25) +
  labs(
    title = "PAM50 subtype distribution in training and test sets",
    x = "PAM50 subtype", y = "Count"
  )

---

## 4. Per-View Distributions

### 4.1 mRNA

In [ ]:
# These 200 mRNA features are pre-selected by mixOmics as the most
# discriminant transcripts. Higher variance = more informative for DIABLO.

mrna_var <- apply(X_train$mRNA, 2, var)

data.frame(variance = mrna_var) |>
  ggplot(aes(x = variance)) +
  geom_histogram(bins = 40, fill = "#4393c3", colour = "white", alpha = 0.85) +
  labs(
    title = "mRNA feature variance distribution (training set)",
    subtitle = sprintf("%d pre-selected discriminant transcripts", ncol(X_train$mRNA)),
    x = "Variance", y = "Count"
  )

In [ ]:
# Heatmap of top 50 most variable mRNA features.
# Good DIABLO input: subtype-specific expression blocks should be visible.
# If the heatmap shows clear subtype clusters, the signal is there.

top50_mrna <- names(sort(mrna_var, decreasing = TRUE)[1:50])
anno_col   <- data.frame(Subtype = Y_train, row.names = rownames(X_train$mRNA))

pheatmap(
  t(X_train$mRNA[, top50_mrna]),
  annotation_col    = anno_col,
  annotation_colors = list(Subtype = subtype_colors),
  scale             = "row",
  clustering_method = "ward.D2",
  show_colnames     = FALSE,
  fontsize_row      = 7,
  main              = "Top 50 variable mRNA features — PAM50 subtype annotation"
)

### 4.2 miRNA

In [ ]:
# microRNAs post-transcriptionally repress gene expression by binding 3'UTR.
# Key miRNAs in breast cancer subtypes:
#   miR-200 family (miR-200a/b/c, miR-141, miR-429): high in luminal subtypes
#     → suppress ZEB1/ZEB2 and SNAI2 (EMT drivers) → maintain epithelial phenotype
#   miR-21: oncomiR, promotes tumour survival → higher in aggressive subtypes
#   let-7 family: tumour suppressors → often low in Basal

mirna_var <- apply(X_train$miRNA, 2, var)

data.frame(mirna = names(mirna_var), variance = mirna_var) |>
  slice_max(variance, n = 20) |>
  ggplot(aes(x = reorder(mirna, variance), y = variance)) +
  geom_col(fill = "#74c476", alpha = 0.85) +
  coord_flip() +
  labs(
    title = "Top 20 most variable miRNA features",
    x = NULL, y = "Variance"
  )

### 4.3 Proteomics (RPPA)

In [ ]:
# RPPA measures functional protein abundance + phosphorylation states.
# Unlike mRNA, RPPA captures post-translational modifications that
# directly reflect signalling pathway activity.
#
# Expected top proteins:
#   HER2/ERBB2 → highest variance due to HER2 subtype overexpression
#   ER/ESR1    → high in luminal, near-zero in basal
#   KRT5       → Basal marker (cytokeratin 5)
#   Ki67/MKI67 → proliferation, high in Basal

prot_var <- apply(X_train$proteomics, 2, var)

data.frame(protein = names(prot_var), variance = prot_var) |>
  slice_max(variance, n = 20) |>
  ggplot(aes(x = reorder(protein, variance), y = variance)) +
  geom_col(fill = "#9970ab", alpha = 0.85) +
  coord_flip() +
  labs(
    title = "Top 20 most variable proteins (RPPA)",
    subtitle = "ER, HER2 expected — PAM50 defining markers",
    x = NULL, y = "Variance"
  )

---

## 5. Single-View PCA per Subtype

In [ ]:
# PCA per view shows how well each single omic layer separates subtypes.
# This motivates DIABLO: if any single view gave perfect separation,
# multi-omics integration would add no value. The mixed-but-imperfect
# separation observed here sets the stage for integration.

run_pca <- function(mat, Y, view_name) {
  pca <- prcomp(mat, scale. = TRUE, center = TRUE)
  var_pct <- round(100 * pca$sdev^2 / sum(pca$sdev^2), 1)
  data.frame(
    PC1 = pca$x[, 1], PC2 = pca$x[, 2],
    subtype = Y, view = view_name,
    pct1 = var_pct[1], pct2 = var_pct[2]
  )
}

pca_df <- bind_rows(
  run_pca(X_train$mRNA,       Y_train, "mRNA"),
  run_pca(X_train$miRNA,      Y_train, "miRNA"),
  run_pca(X_train$proteomics, Y_train, "proteomics")
)

ggplot(pca_df, aes(x = PC1, y = PC2, colour = subtype)) +
  geom_point(size = 2, alpha = 0.75) +
  facet_wrap(~ view, scales = "free") +
  scale_colour_manual(values = subtype_colors) +
  labs(
    title    = "Single-view PCA: partial PAM50 subtype separation per omic",
    subtitle = "Partial separation in each view motivates multi-omics integration",
    colour   = "PAM50 subtype"
  ) +
  theme(legend.position = "bottom")

---

## 6. Known PAM50 Marker Validation

In [ ]:
# Verify data quality: known PAM50 marker genes should show expected
# subtype-specific expression. If they don't, data processing may be wrong.
#
# ESR1 → high in LumA (ER-positive luminal tumours)
# ERBB2 → high in Her2 (HER2-amplified tumours)
# CCNB1/MKI67 → high in Basal (highly proliferative)

known_genes <- intersect(
  c("ESR1", "ERBB2", "MKI67", "CCNB1", "FOXA1", "GATA3"),
  colnames(X_train$mRNA)
)

if (length(known_genes) >= 2) {
  as.data.frame(X_train$mRNA[, known_genes, drop = FALSE]) |>
    mutate(subtype = Y_train) |>
    pivot_longer(cols = all_of(known_genes), names_to = "gene",
                 values_to = "expression") |>
    ggplot(aes(x = subtype, y = expression, fill = subtype)) +
    geom_violin(alpha = 0.7, trim = TRUE) +
    geom_jitter(width = 0.15, size = 0.8, alpha = 0.4) +
    facet_wrap(~ gene, scales = "free_y") +
    scale_fill_manual(values = subtype_colors) +
    labs(
      title    = "Known PAM50 markers — subtype expression validation",
      subtitle = "Expected: ESR1/FOXA1 high in LumA; ERBB2 high in Her2; MKI67/CCNB1 high in Basal",
      x = NULL, y = "Expression", fill = "Subtype"
    ) +
    theme(legend.position = "bottom")
} else {
  cat("Known marker genes not in the 200-gene set; showing top variable mRNA by subtype.\n")

  as.data.frame(X_train$mRNA[, names(sort(mrna_var, TRUE)[1:4])]) |>
    mutate(subtype = Y_train) |>
    pivot_longer(-subtype, names_to = "gene", values_to = "expression") |>
    ggplot(aes(x = subtype, y = expression, fill = subtype)) +
    geom_violin(alpha = 0.7) +
    facet_wrap(~ gene, scales = "free_y") +
    scale_fill_manual(values = subtype_colors) +
    labs(title = "Top variable mRNA by subtype (fallback)", x = NULL, y = "Expression")
}

---

## 7. Cross-View Correlation Structure

In [ ]:
# Pearson correlation between PC1 of each view.
# This informs the DIABLO design matrix (Notebook 3):
#   High cross-view correlation → design value can be 1 (maximise correlation)
#   Low correlation → use 0.1 (prioritise discrimination; default recommendation)

pc1_views <- map2(
  X_train, names(X_train),
  ~ setNames(data.frame(prcomp(.x, scale. = TRUE)$x[, 1]), .y)
) |> bind_cols()

cor_mat <- cor(pc1_views)

corrplot(
  cor_mat,
  method   = "number",
  type     = "upper",
  tl.col   = "black",
  title    = "Cross-view correlation (PC1 per block)",
  mar      = c(0, 0, 2, 0),
  col      = colorRampPalette(c("#4575b4", "white", "#d73027"))(100)
)

---

## 8. Save Processed Data

In [ ]:
dir.create("../../results/diablo", recursive = TRUE, showWarnings = FALSE)

saveRDS(
  list(X_train = X_train, Y_train = Y_train,
       X_test  = X_test,  Y_test  = Y_test),
  "../../results/diablo/breast_TCGA_processed.RDS"
)

cat("Saved to results/diablo/breast_TCGA_processed.RDS\n")

---

## Summary

| Block | Features | Train samples | Test samples |
|-------|---------|--------------|-------------|
| mRNA | `r ncol(X_train$mRNA)` | `r nrow(X_train$mRNA)` | `r nrow(X_test$mRNA)` |
| miRNA | `r ncol(X_train$miRNA)` | `r nrow(X_train$miRNA)` | `r nrow(X_test$miRNA)` |
| proteomics | `r ncol(X_train$proteomics)` | `r nrow(X_train$proteomics)` | 0 (not in test split) |

**Subtype counts (train)**: `r paste(names(table(Y_train)), table(Y_train), sep="=", collapse=", ")`

**Key EDA observations**:
- All three views show partial but imperfect subtype separation in PCA
- mRNA gives the clearest Basal vs Luminal split; miRNA and proteomics add orthogonal signal
- Value-level missingness is minimal within available matrices
- Proteomics has block-level missingness in test data (entire block unavailable)
- No severe class imbalance; DIABLO should train fairly across subtypes
- Known PAM50 markers (ESR1, ERBB2, Ki67) show expected expression patterns → data quality confirmed

**→ Next: `02_single_omics_baseline.Rmd`**